In [1]:
# Copyright (c) 2024, NVIDIA CORPORATION.
"""Pretrain Vision Transformer using Megatron Core."""
import os
import math
import torch
import torch.nn.functional as F
from torch import Tensor

from megatron.core import parallel_state, tensor_parallel
from megatron.core.enums import ModelType
from megatron.core.transformer.transformer_block import TransformerBlock
from megatron.core.transformer.transformer_config import TransformerConfig
from megatron.core.transformer.module import GraphableMegatronModule, MegatronModule
from megatron.core.models.vision.vit_layer_specs import (
    get_vit_layer_with_local_spec,
    get_vit_layer_with_transformer_engine_spec,
)
from megatron.core.models.common.embeddings import RotaryEmbeddingDinoV3

from megatron.training import (
    get_args,
    get_timers,
    pretrain,
    print_rank_0,
)
from megatron.training.arguments import core_transformer_config_from_args

from megatron.core.datasets.blended_megatron_dataset_builder import (
    BlendedMegatronDatasetBuilder,
)
from megatron.core.datasets.utils import Split
from megatron.core.datasets.blended_megatron_dataset_config import (
    BlendedMegatronDatasetConfig,
)
from megatron.training import initialize_megatron


import sys, shlex

with open("debug_args.txt") as f:
    args_list = shlex.split(f.read())

sys.argv = ["notebook"] + args_list


/usr/local/lib/python3.12/dist-packages/torch/library.py:356: UserWarning: Warning only once for all operators,  other operators may also be overridden.
  Overriding a previously registered kernel for the same operator and the same dispatch key
  operator: flash_attn::_flash_attn_backward(Tensor dout, Tensor q, Tensor k, Tensor v, Tensor out, Tensor softmax_lse, Tensor(a6!)? dq, Tensor(a7!)? dk, Tensor(a8!)? dv, float dropout_p, float softmax_scale, bool causal, SymInt window_size_left, SymInt window_size_right, float softcap, Tensor? alibi_slopes, bool deterministic, Tensor? rng_state=None) -> Tensor
    registered at /usr/local/lib/python3.12/dist-packages/torch/_library/custom_ops.py:922
  dispatch key: ADInplaceOrView
  previous kernel: no debug info
       new kernel: registered at /usr/local/lib/python3.12/dist-packages/torch/_library/custom_ops.py:922 (Triggered internally at /opt/pytorch/pytorch/aten/src/ATen/core/dispatch/OperatorEntry.cpp:208.)
  self.m.impl(
/workspace/Trans

In [2]:
from pretrain_vision import *

In [3]:
initialize_megatron(
        extra_args_provider=add_vit_args,
        args_defaults={},
        get_embedding_ranks=None,
        get_position_embedding_ranks=None,
        store=None,
    )

using world size: 1, data-parallel size: 1, context-parallel size: 1, hierarchical context-parallel sizes: None, tensor-model-parallel size: 1, pipeline-model-parallel size: 1
Number of virtual stages per pipeline stage: None
accumulate and all-reduce gradients in fp32 for bfloat16 data type.
using torch.bfloat16 for parameters ...
------------------------ arguments ------------------------
  account_for_embedding_in_pipeline_split ......... False
  account_for_loss_in_pipeline_split .............. False
  accumulate_allreduce_grads_in_fp32 .............. True
  activation_func_clamp_value ..................... None
  adam_beta1 ...................................... 0.9
  adam_beta2 ...................................... 0.95
  adam_eps ........................................ 1e-08
  add_bias_linear ................................. True
  add_position_embedding .......................... True
  add_qkv_bias .................................... True
  adlr_autoresume ................

/usr/local/lib/python3.12/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
[rank0]:[W323 19:11:17.517617069 ProcessGroupNCCL.cpp:5156] Guessing device ID based on global rank. This can cause a hang if rank to GPU mapping is heterogeneous. You can specify device_id in init_process_group()


In [4]:
model = model_provider(pre_process=True, post_process=True,config=None,pg_collection=None,wrap_with_ddp=False)

Building ViT model ...


height:512, width:512




patch size: 16



dim at rope: 64



base frequency: 100.0, self.dim: 64
inv_freq device: cpu


periods size: torch.Size([16])



/workspace/megatron-lm/megatron/core/transformer/transformer_config.py:1189: UserWarning: If you are using transformer_engine as the transformer implementation, the core_attn is from transformer_engine and may be the fused version. For fused attention, you have no need to set 'core_attn' to recompute. Please check that the core_attn recompute is really needed.
  warnings.warn(


In [5]:
train_ds, valid_ds, test_ds = train_valid_test_datasets_provider()

Building ViT datasets with BlendedMegatronDatasetBuilder ...
INFO:megatron.core.datasets.blended_megatron_dataset_config:blend not provided for test split
INFO:megatron.core.datasets.blended_megatron_dataset_builder:Building MegatronVisionDataset splits with sizes=[20, 1, 1] and config=BlendedMegatronDatasetConfig(random_seed=1234, sequence_length=0, image_size=512, blend=None, blend_per_split=[(['/workspace/dataset/training'], None), (['/workspace/dataset/testing'], None), None], multiple_validation_sets=None, full_validation=None, split=None, split_matrix=None, num_dataset_builder_threads=1, path_to_cache=None, mmap_bin_files=True, mock=False, tokenizer=None, mid_level_dataset_surplus=0.005, allow_ambiguous_pad_tokens=False, fast_cache_load=False, defer_npy_index_mmap=False)


In [6]:
image = train_ds[0]["images"]
label = train_ds[0]["labels"]
image = torch.unsqueeze(image, 0)
image = image.to("cuda",dtype=torch.bfloat16)
label = label.to("cuda")



In [7]:
image.to("cuda",)

tensor([[[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]],

         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]],

         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]]]], device='cuda:0',
       dtype=torch.bfloat16)

In [8]:
from pretrain_vision import *
model = model_provider(pre_process=True, post_process=True,config=None,pg_collection=None,wrap_with_ddp=False)
meg_model = model.to("cuda",dtype=torch.bfloat16)
meg_model(image)

Building ViT model ...


height:512, width:512




patch size: 16



dim at rope: 64



base frequency: 100.0, self.dim: 64
inv_freq device: cpu


periods size: torch.Size([16])



x size: torch.Size([1, 384, 32, 32])



angles size: torch.Size([1024, 2, 16])


angles size after flatten: torch.Size([1024, 32])


angles sizeafter branch: torch.Size([1024, 64])



 embed size: torch.Size([1024, 128])




pos_emb: torch.Size([1024, 1, 1, 64]), x: torch.Size([1025, 1, 384])


hidden_states at attention forward: torch.Size([1025, 1, 384])
q_pos_emb size: torch.Size([1024, 1, 1, 64]), k_pos_emb size: torch.Size([1024, 1, 1, 64])
hf query size at mlm rope: torch.Size([1029, 1, 6, 64])


query shape: torch.Size([1029, 1, 6, 64]), num_prefix_tokens: 1, num_pos_tokens: 1024


query shape after split: torch.Size([1024, 1, 6, 64]), pos_emb size: torch.Size([1024, 1, 1, 64])
_apply_rotary_pos_emb_bshd
counter: 1, rotary_interleaved: False
mscale: 1.0
_apply_rotary_pos_emb_bshd
counter: 2, rotary_in

AssertionError: 

In [9]:
import torch
from transformers import AutoImageProcessor, AutoModel
from transformers.image_utils import load_image

pretrained_model_name = "facebook/dinov3-vits16-pretrain-lvd1689m"
processor = AutoImageProcessor.from_pretrained(pretrained_model_name)
model = AutoModel.from_pretrained(
    pretrained_model_name, 
    device_map="auto", 
    dtype=torch.bfloat16,
)

INFO:httpx:HTTP Request: HEAD https://huggingface.co/facebook/dinov3-vits16-pretrain-lvd1689m/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/facebook/dinov3-vits16-pretrain-lvd1689m/resolve/main/preprocessor_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/facebook/dinov3-vits16-pretrain-lvd1689m/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/facebook/dinov3-vits16-pretrain-lvd1689m/resolve/main/preprocessor_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/facebook/dinov3-vits16-pretrain-lvd1689m/resolve/main/config.json "HTTP/1.1 200 OK"

dim at rope: 64



base frequency: 100.0, self.head: 64
inv_freq device: meta


inv_freq size: torch.Size([16])



Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

In [10]:
out = model(image)
print(model)

64
inv_freq: tensor([1.0000, 0.7499, 0.5623, 0.4217, 0.3162, 0.2371, 0.1778, 0.1334, 0.1000,
        0.0750, 0.0562, 0.0422, 0.0316, 0.0237, 0.0178, 0.0133],
       device='cuda:0')

angles size: torch.Size([1024, 2, 16])


angles size after flatten: torch.Size([1024, 32])


angles size after tile: torch.Size([1024, 64])



 cos size: torch.Size([1024, 64]), sin size: torch.Size([1024, 64])


real_hidden_states size: torch.Size([1, 1029, 384])

hidden states size: torch.Size([1, 1029, 384])

pos emb sizes: torch.Size([1024, 64]), torch.Size([1024, 64])


AssertionError: 

In [ ]:
pos_emb_hf = torch.load("/workspace/megatron-lm/hf_emb_tensor")
pos_emb_mlm = torch.load("/workspace/megatron-lm/mlm_emb_tensor")

In [11]:
files = [
    "attention-inputs",
    "attention-output",
    "cos_sin_cat_emb",
    "key-rope-inputs",
    "key-rope-outputs",
    "query-rope-inputs",
    "query-rope-outputs",
    "angles",
    "coords",
    "periods",
    "periods_init",
    "rotate_half_tokens_q",
    "rotate_half_tokens_k",
]

In [ ]:
for name in files:
    mlm_tensor = torch.load(f"/workspace/megatron-lm/mlm-tensors/{name}")
    hf_tensor = torch.load(f"/workspace/megatron-lm/hf-tensors/{name}")
    if name not in ["cos_sin_cat_emb","angles","coords","periods","periods_init",
                    "rotate_half_tokens_q","rotate_half_tokens_k"]:
        hf_tensor = hf_tensor.transpose(0,1)
        if name in ["key-rope-inputs",
            "key-rope-outputs",
            "query-rope-inputs",
            "query-rope-outputs"]:
            hf_tensor = hf_tensor.transpose(0,2)
        cls_token, _, hf_tokens = hf_tensor.split([1,4,1024], dim=0)
        hf_tensor = torch.cat((cls_token,hf_tokens), dim=0)
    elif name in ["rotate_half_tokens_q","rotate_half_tokens_k"]:
        hf_tensor = hf_tensor.transpose(0,2)
        pass
        #print(mlm_tensor, "\n")
        #print(hf_tensor, "\n")
    if name in ["key-rope-inputs",
        "key-rope-outputs",
        "query-rope-inputs",
        "query-rope-outputs"]:
        hf_tensor = hf_tensor.transpose(1,2)
    print(f"sizes of {name}: {mlm_tensor.size()}, {hf_tensor.size()}")
    diff = (mlm_tensor.squeeze() - hf_tensor.squeeze()).abs()
    print(name)
    print(f"Max diff:", diff.max())
    print("Mean diff:", diff.mean())

sizes of attention-inputs: torch.Size([1025, 1, 384]), torch.Size([1025, 1, 384])
attention-inputs
Max diff: tensor(54.7500, dtype=torch.bfloat16, grad_fn=<MaxBackward1>)
Mean diff: tensor(3.9219, dtype=torch.bfloat16, grad_fn=<MeanBackward0>)
sizes of attention-output: torch.Size([1025, 1, 384]), torch.Size([1025, 1, 384])
attention-output
Max diff: tensor(43.5000, dtype=torch.bfloat16, grad_fn=<MaxBackward1>)
Mean diff: tensor(1.6875, dtype=torch.bfloat16, grad_fn=<MeanBackward0>)
sizes of cos_sin_cat_emb: torch.Size([1024, 128]), torch.Size([1024, 128])
cos_sin_cat_emb
Max diff: tensor(0.)
Mean diff: tensor(0.)
sizes of key-rope-inputs: torch.Size([1025, 1, 6, 64]), torch.Size([1025, 6, 1, 64])
key-rope-inputs
Max diff: tensor(724., dtype=torch.bfloat16, grad_fn=<MaxBackward1>)
Mean diff: tensor(217., dtype=torch.bfloat16, grad_fn=<MeanBackward0>)
sizes of key-rope-outputs: torch.Size([1025, 1, 6, 64]), torch.Size([1025, 6, 1, 64])
key-rope-outputs
Max diff: tensor(0., dtype=torch.b

IndexError: Dimension out of range (expected to be in range of [-1, 0], but got 1)

In [ ]:
name = "angles"
mlm_tensor = torch.load(f"/workspace/megatron-lm/mlm-tensors/{name}")
hf_tensor = torch.load(f"/workspace/megatron-lm/hf-tensors/{name}")
print(mlm_tensor)
print(hf_tensor)

tensor([[-6.0868, -4.5651, -3.4238,  ..., -0.1441, -0.1085, -0.0810],
        [-6.0868, -4.5651, -3.4238,  ..., -0.1348, -0.1015, -0.0758],
        [-6.0868, -4.5651, -3.4238,  ..., -0.1255, -0.0945, -0.0705],
        ...,
        [ 6.0868,  4.5651,  3.4238,  ...,  0.1255,  0.0945,  0.0705],
        [ 6.0868,  4.5651,  3.4238,  ...,  0.1348,  0.1015,  0.0758],
        [ 6.0868,  4.5651,  3.4238,  ...,  0.1441,  0.1085,  0.0810]])
tensor([[-6.0868, -4.5645, -3.4229,  ..., -0.1443, -0.1082, -0.0812],
        [-6.0868, -4.5645, -3.4229,  ..., -0.1350, -0.1013, -0.0759],
        [-6.0868, -4.5645, -3.4229,  ..., -0.1257, -0.0943, -0.0707],
        ...,
        [ 6.0868,  4.5645,  3.4229,  ...,  0.1257,  0.0943,  0.0707],
        [ 6.0868,  4.5645,  3.4229,  ...,  0.1350,  0.1013,  0.0759],
        [ 6.0868,  4.5645,  3.4229,  ...,  0.1443,  0.1082,  0.0812]])


In [ ]:
print(1/mlm_tensor)
print(hf_tensor)

tensor([1.0000, 0.7500, 0.5625, 0.4219, 0.3164, 0.2373, 0.1777, 0.1338, 0.1001,
        0.0752, 0.0564, 0.0420, 0.0317, 0.0237, 0.0178, 0.0133],
       dtype=torch.bfloat16)
tensor([1.0000, 0.7499, 0.5623, 0.4217, 0.3162, 0.2371, 0.1778, 0.1334, 0.1000,
        0.0750, 0.0562, 0.0422, 0.0316, 0.0237, 0.0178, 0.0133])


In [ ]:
inputs_hf.transpose(1,2).transpose(0,1).size()

torch.Size([1025, 1, 6, 64])

In [ ]:
inputs_mlm - inputs_hf.transpose(1,2).transpose(0,1)

tensor([[[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]]],


        [[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]]],


        [[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]]],


        ...,


        [[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
       

In [ ]:
outputs_hf = torch.load("/workspace/megatron-lm/query-outputs-hf")
outputs_mlm = torch.load("/workspace/megatron-lm/query-outputs-mlm")

In [ ]:
outputs_hf.transpose(1,2).transpose(0,1)

tensor([[[[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]]],


        [[[ 0.0074,  0.1406,  0.1299,  ..., -0.0879,  0.0796, -0.1445],
          [ 0.0552,  0.0012,  0.0300,  ...,  0.2314,  0.0515, -0.0139],
          [-0.1406, -0.0221,  0.1094,  ..., -0.1533, -0.0566, -0.0977],
          [-0.0300,  0.0305,  0.2188,  ..., -0.2100,  0.0020, -0.0574],
          [-0.0649, -0.0801,  0.1367,  ..., -0.0171,  0.0258, -0.0143],
          [ 0.0293, -0.2363, -0.0488,  ...,  0.1777, -0.0869,  0.2383]]],


        [[[ 0.0074,  0.1406,  0.1299,  ..., -0.0898,  0.0801, -0.1445],
          [ 0.0552,  0.0012,  0.0300,  ...,  0.2295,  0.

In [ ]:
outputs_mlm

tensor([[[[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]]],


        [[[ 0.0074,  0.1426,  0.1309,  ..., -0.0879,  0.0796, -0.1455],
          [ 0.0552,  0.0021,  0.0298,  ...,  0.2314,  0.0513, -0.0139],
          [-0.1406, -0.0212,  0.1094,  ..., -0.1533, -0.0569, -0.0977],
          [-0.0302,  0.0308,  0.2197,  ..., -0.2109,  0.0020, -0.0574],
          [-0.0649, -0.0815,  0.1357,  ..., -0.0172,  0.0258, -0.0144],
          [ 0.0293, -0.2373, -0.0488,  ...,  0.1777, -0.0869,  0.2383]]],


        [[[ 0.0074,  0.1426,  0.1309,  ..., -0.0898,  0.0806, -0.1445],
          [ 0.0552,  0.0021,  0.0298,  ...,  0.2295,  0.

In [ ]:
torch.sum(torch.abs(outputs_mlm - outputs_hf.transpose(1,2).transpose(0,1)) > -10000)

tensor(393600)

In [ ]:
outputs_hf = torch.load("/workspace/megatron-lm/key-outputs-hf")
outputs_mlm = torch.load("/workspace/megatron-lm/key-outputs-mlm")

In [ ]:
torch.sum(torch.abs(outputs_mlm - outputs_hf.transpose(1,2).transpose(0,1)) > 10e-4)

tensor(6780)

In [ ]:
outputs_hf.transpose(1,2).transpose(0,1)

tensor([[[[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
            0.0000e+00,  0.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
            0.0000e+00,  0.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
            0.0000e+00,  0.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
            0.0000e+00,  0.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
            0.0000e+00,  0.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
            0.0000e+00,  0.0000e+00]]],


        [[[ 1.1475e-01, -1.6211e-01,  1.3062e-02,  ..., -7.0190e-03,
           -3.6328e-01, -1.9727e-01],
          [ 1.7578e-01, -1.6211e-01, -8.2031e-02,  ...,  2.3438e-01,
            1.7969e-01,  4.8828e-04],
          [-4.9316e-02,  5.9082e-02,  4.4922e-02,  ...,  1.2500e-01,
            8.3008e-02, -1.0986e-01],
          [ 5.3223e-02, -5.2246e-